# 전략 워크벤치 Colab 실행기

이 노트북은 여러 전략을 한 번에 비교하면서, 기간/이평선/레버리지/분할익절/트레일링 스탑까지 직접 바꿔볼 수 있게 만든 실행용 노트북입니다.

사용 순서

1. 아래 `공통 설정`, `전략 1`, `전략 2`, `전략 3` 입력칸을 채웁니다.
2. 셀을 위에서부터 순서대로 실행합니다.
3. 결과표와 비교 차트를 확인합니다.
4. 마지막 셀에서 결과 파일을 zip으로 내려받을 수 있습니다.

중요:
- 이번 버전은 사용자가 보는 입력칸 이름을 최대한 한글로 바꾼 버전입니다.
- 드롭다운 값은 한글로 골라도 내부적으로 자동 변환되어 기존 엔진 형식으로 실행됩니다.


In [ ]:
REPO_URL = "https://github.com/snowballTQ/multi-strategy-fucker.git"
BRANCH = "main"


In [ ]:
from pathlib import Path
import shutil
import subprocess

repo_dir = Path('/content/strategy_workbench_repo')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    'git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(repo_dir)
], check=True)

print(f'Repo directory: {repo_dir}')


## 입력 규칙 안내

이번 Colab은 `config 딕셔너리`를 직접 만지지 않고, 아래 폼 입력만으로 전략을 비교할 수 있게 만든 버전입니다.

입력 원칙은 단순합니다.

- 날짜는 달력에서 고르면 됩니다.
- 숫자는 직접 입력하면 됩니다.
- 글자로 고르는 항목은 드롭다운으로 준비했습니다.
- 전략은 1~3번 슬롯 중 켜고 싶은 것만 사용하면 됩니다.

### 1. 공통 설정

- `시작일`, `종료일`: 이번 비교를 어느 구간에서 돌릴지 정합니다.
- `편도수수료`: 예: `0.001` = 0.10%
- `세율`: 예: `0.22` = 22%
- `연운용보수`: 예: `0.0095` = 0.95%
- `차입스프레드`: 차입 비용 가정값입니다. 예: `1.0`
- `결과폴더이름`: 실험마다 결과 폴더 이름을 다르게 두면 비교하기 편합니다.

### 2. 전략 슬롯

- `전략1_사용`, `전략2_사용`, `전략3_사용`이 각 슬롯의 켜기/끄기입니다.
- 하나만 켜면 단일 전략 테스트, 둘 이상 켜면 같은 조건에서 동시 비교가 됩니다.
- `전략X_이름`은 결과표와 차트에 보일 이름입니다.
- `전략X_기준시계열`, `전략X_레버리지`, `전략X_비용반영`, `전략X_전량청산후자금`이 전략의 기본 뼈대입니다.

### 3. 드롭다운 값 의미

- `기준시계열`
  - `나스닥100`: NDX 기준
  - `S&P500`: SPX 기준
  - `합성장기시계열`: composite splice 기준
- `전량청산후자금`, `익절 자금이동`, `트레일링 자금이동`
  - `현금`, `SGOV`, `SPY`, `QQQ`, `기준시계열` 중 선택합니다.
- `진입결합`, `청산결합`
  - `규칙1만`: 규칙1만 사용
  - `둘다만족`: 규칙1과 규칙2를 동시에 만족해야 함
  - `하나만만족`: 규칙1 또는 규칙2 중 하나만 만족하면 됨
- `규칙 유형`
  - `가격>SMA`, `가격<SMA`: 가격과 이동평균 비교
  - `빠른선>느린선`, `빠른선<느린선`: 두 이동평균 비교
  - `이평선정배열`, `이평선역배열`: 3개 이상 이평선 순서 비교
  - `항상참`: 조건 없이 항상 true
  - `사용안함`: 두 번째 규칙을 끌 때 사용
- `규칙 기준`
  - `매매대상`: 실제 매매되는 레버리지 시계열 기준
  - `기준시계열`: 해당 전략의 base series 기준
  - `나스닥100`, `S&P500`, `합성장기시계열`: 특정 지수를 직접 기준으로 사용

### 4. 기간칸 입력법

- `가격>SMA`, `가격<SMA`
  - `기간1`만 씁니다.
  - 예: `기간1 = 200`
- `빠른선>느린선`, `빠른선<느린선`
  - `기간1 = 빠른선`, `기간2 = 느린선`
  - 예: `50`, `200`
- `이평선정배열`, `이평선역배열`
  - `기간1 / 기간2 / 기간3`을 순서대로 씁니다.
  - 예: `3`, `161`, `185`
- `사용안함`, `항상참`
  - 기간칸은 무시되므로 `0`으로 둬도 됩니다.

### 5. 부분익절 입력법

- `1차익절_사용`, `2차익절_사용`, `3차익절_사용`으로 각 단계 on/off를 고릅니다.
- `배수`는 진입가 대비 목표 배수입니다.
  - 예: `1.15` = +15%, `1.68` = +68%, `2.00` = +100%
- `매도비율`은 현재 보유분 중 얼마를 팔지 정하는 값입니다.
  - 예: `0.50` = 50%, `0.35` = 35%, `1.00` = 전량
- `자금이동`은 익절 대금을 어디로 보낼지 정합니다.

### 6. 트레일링 스탑 입력법

- `전략X_트레일링사용`을 켜면 트레일링 스탑이 작동합니다.
- `전략X_트레일링_첫익절후활성 = True`면 첫 익절 이후부터만 트레일링을 켭니다.
- `전략X_트레일링_하락폭`
  - 예: `0.15` = 고점 대비 15% 하락 시 전량 청산
- `전략X_트레일링_자금이동`은 청산 대금 목적지입니다.

### 7. 바로 써먹는 예시

- 200일선 가격 전략
  - 진입규칙1 유형: `가격>SMA`
  - 진입규칙1 기간1: `200`
  - 청산규칙1 유형: `가격<SMA`
  - 청산규칙1 기간1: `200`
  - 레버리지: `3.0`
- 3-161-185 조합 + 부분익절
  - 진입결합: `둘다만족`
  - 진입규칙1: `가격>SMA`, 기간1 = `200`
  - 진입규칙2: `이평선정배열`, 기간1/2/3 = `3`, `161`, `185`
  - 청산결합: `하나만만족`
  - 청산규칙1: `가격<SMA`, 기간1 = `200`
  - 청산규칙2: `이평선역배열`, 기간1/2/3 = `3`, `161`, `185`
  - 1차익절 배수 `1.15`, 매도비율 `0.50`
  - 2차익절 배수 `1.68`, 매도비율 `0.35`
- 듀얼 이평선 교차 전략
  - 진입규칙1 유형: `빠른선>느린선`, 기간1/2 = `50`, `200`
  - 청산규칙1 유형: `빠른선<느린선`, 기간1/2 = `50`, `200`

### 8. 자주 쓰는 숫자 예시

- 이평선: `3`, `5`, `20`, `50`, `60`, `120`, `161`, `185`, `200`
- 레버리지: `1.0`, `2.0`, `3.0`
- 익절 배수: `1.10`, `1.15`, `1.25`, `1.50`, `1.68`, `2.00`
- 트레일링 하락폭: `0.10`, `0.15`, `0.20`

### 9. 처음엔 이렇게 해보시면 편합니다

1. 우선 기본값 그대로 한 번 실행합니다.
2. 그다음 전략 3을 켜서 비교 전략을 추가합니다.
3. 그다음 레버리지, 기간1/2/3, 익절 배수만 조금씩 바꿔봅니다.
4. 마지막으로 자금이동과 트레일링 스탑을 켜서 차이를 봅니다.

실수 방지 팁

- 규칙2를 안 쓸 거면 `진입규칙2_유형`, `청산규칙2_유형`을 `사용안함`으로 두면 됩니다.
- `빠른선>느린선`, `빠른선<느린선`은 보통 `기간1 < 기간2`가 자연스럽습니다.
- `이평선정배열`은 짧은 이평선부터 긴 이평선 순으로 넣는 쪽이 읽기 쉽습니다.
- 처음에는 전략 2개만 켜고 돌려본 뒤, 그다음 3개 비교로 늘리는 것을 추천합니다.


In [ ]:
# @title 공통 설정
시작일 = "1985-10-01" # @param {type:"date"}
종료일 = "2026-03-09" # @param {type:"date"}
편도수수료 = 0.001 # @param {type:"number"}
세율 = 0.22 # @param {type:"number"}
연운용보수 = 0.0095 # @param {type:"number"}
차입스프레드 = 1.0 # @param {type:"number"}
결과폴더이름 = "strategy_workbench_demo" # @param {type:"string"}


In [ ]:
# @title 전략 1
전략1_사용 = True # @param {type:"boolean"}
전략1_이름 = "200일선 가격 전략 | 3배" # @param {type:"string"}
전략1_기준시계열 = "나스닥100" # @param ["나스닥100", "S&P500", "합성장기시계열"]
전략1_레버리지 = 3.0 # @param {type:"number"}
전략1_비용반영 = True # @param {type:"boolean"}
전략1_전량청산후자금 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략1_진입결합 = "규칙1만" # @param ["규칙1만", "둘다만족", "하나만만족"]
전략1_진입규칙1_유형 = "가격>SMA" # @param ["가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략1_진입규칙1_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략1_진입규칙1_기간1 = 200 # @param {type:"integer"}
전략1_진입규칙1_기간2 = 0 # @param {type:"integer"}
전략1_진입규칙1_기간3 = 0 # @param {type:"integer"}
전략1_진입규칙2_유형 = "사용안함" # @param ["사용안함", "가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략1_진입규칙2_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략1_진입규칙2_기간1 = 0 # @param {type:"integer"}
전략1_진입규칙2_기간2 = 0 # @param {type:"integer"}
전략1_진입규칙2_기간3 = 0 # @param {type:"integer"}
전략1_청산결합 = "규칙1만" # @param ["규칙1만", "둘다만족", "하나만만족"]
전략1_청산규칙1_유형 = "가격<SMA" # @param ["가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략1_청산규칙1_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략1_청산규칙1_기간1 = 200 # @param {type:"integer"}
전략1_청산규칙1_기간2 = 0 # @param {type:"integer"}
전략1_청산규칙1_기간3 = 0 # @param {type:"integer"}
전략1_청산규칙2_유형 = "사용안함" # @param ["사용안함", "가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략1_청산규칙2_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략1_청산규칙2_기간1 = 0 # @param {type:"integer"}
전략1_청산규칙2_기간2 = 0 # @param {type:"integer"}
전략1_청산규칙2_기간3 = 0 # @param {type:"integer"}
전략1_1차익절_사용 = False # @param {type:"boolean"}
전략1_1차익절_배수 = 1.15 # @param {type:"number"}
전략1_1차익절_매도비율 = 0.5 # @param {type:"number"}
전략1_1차익절_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략1_2차익절_사용 = False # @param {type:"boolean"}
전략1_2차익절_배수 = 1.68 # @param {type:"number"}
전략1_2차익절_매도비율 = 0.35 # @param {type:"number"}
전략1_2차익절_자금이동 = "SPY" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략1_3차익절_사용 = False # @param {type:"boolean"}
전략1_3차익절_배수 = 2.0 # @param {type:"number"}
전략1_3차익절_매도비율 = 1.0 # @param {type:"number"}
전략1_3차익절_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략1_트레일링사용 = False # @param {type:"boolean"}
전략1_트레일링_첫익절후활성 = True # @param {type:"boolean"}
전략1_트레일링_하락폭 = 0.15 # @param {type:"number"}
전략1_트레일링_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]


In [ ]:
# @title 전략 2
전략2_사용 = True # @param {type:"boolean"}
전략2_이름 = "3-161-185 + 분할익절 | 3배" # @param {type:"string"}
전략2_기준시계열 = "나스닥100" # @param ["나스닥100", "S&P500", "합성장기시계열"]
전략2_레버리지 = 3.0 # @param {type:"number"}
전략2_비용반영 = True # @param {type:"boolean"}
전략2_전량청산후자금 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략2_진입결합 = "둘다만족" # @param ["규칙1만", "둘다만족", "하나만만족"]
전략2_진입규칙1_유형 = "가격>SMA" # @param ["가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략2_진입규칙1_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략2_진입규칙1_기간1 = 200 # @param {type:"integer"}
전략2_진입규칙1_기간2 = 0 # @param {type:"integer"}
전략2_진입규칙1_기간3 = 0 # @param {type:"integer"}
전략2_진입규칙2_유형 = "이평선정배열" # @param ["사용안함", "가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략2_진입규칙2_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략2_진입규칙2_기간1 = 3 # @param {type:"integer"}
전략2_진입규칙2_기간2 = 161 # @param {type:"integer"}
전략2_진입규칙2_기간3 = 185 # @param {type:"integer"}
전략2_청산결합 = "하나만만족" # @param ["규칙1만", "둘다만족", "하나만만족"]
전략2_청산규칙1_유형 = "가격<SMA" # @param ["가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략2_청산규칙1_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략2_청산규칙1_기간1 = 200 # @param {type:"integer"}
전략2_청산규칙1_기간2 = 0 # @param {type:"integer"}
전략2_청산규칙1_기간3 = 0 # @param {type:"integer"}
전략2_청산규칙2_유형 = "이평선역배열" # @param ["사용안함", "가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략2_청산규칙2_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략2_청산규칙2_기간1 = 3 # @param {type:"integer"}
전략2_청산규칙2_기간2 = 161 # @param {type:"integer"}
전략2_청산규칙2_기간3 = 185 # @param {type:"integer"}
전략2_1차익절_사용 = True # @param {type:"boolean"}
전략2_1차익절_배수 = 1.15 # @param {type:"number"}
전략2_1차익절_매도비율 = 0.5 # @param {type:"number"}
전략2_1차익절_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략2_2차익절_사용 = True # @param {type:"boolean"}
전략2_2차익절_배수 = 1.68 # @param {type:"number"}
전략2_2차익절_매도비율 = 0.35 # @param {type:"number"}
전략2_2차익절_자금이동 = "SPY" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략2_3차익절_사용 = False # @param {type:"boolean"}
전략2_3차익절_배수 = 2.0 # @param {type:"number"}
전략2_3차익절_매도비율 = 1.0 # @param {type:"number"}
전략2_3차익절_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략2_트레일링사용 = True # @param {type:"boolean"}
전략2_트레일링_첫익절후활성 = True # @param {type:"boolean"}
전략2_트레일링_하락폭 = 0.15 # @param {type:"number"}
전략2_트레일링_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]


In [ ]:
# @title 전략 3
전략3_사용 = False # @param {type:"boolean"}
전략3_이름 = "듀얼 이평선 교차 | 2배" # @param {type:"string"}
전략3_기준시계열 = "나스닥100" # @param ["나스닥100", "S&P500", "합성장기시계열"]
전략3_레버리지 = 2.0 # @param {type:"number"}
전략3_비용반영 = True # @param {type:"boolean"}
전략3_전량청산후자금 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략3_진입결합 = "규칙1만" # @param ["규칙1만", "둘다만족", "하나만만족"]
전략3_진입규칙1_유형 = "빠른선>느린선" # @param ["가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략3_진입규칙1_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략3_진입규칙1_기간1 = 50 # @param {type:"integer"}
전략3_진입규칙1_기간2 = 200 # @param {type:"integer"}
전략3_진입규칙1_기간3 = 0 # @param {type:"integer"}
전략3_진입규칙2_유형 = "사용안함" # @param ["사용안함", "가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략3_진입규칙2_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략3_진입규칙2_기간1 = 0 # @param {type:"integer"}
전략3_진입규칙2_기간2 = 0 # @param {type:"integer"}
전략3_진입규칙2_기간3 = 0 # @param {type:"integer"}
전략3_청산결합 = "규칙1만" # @param ["규칙1만", "둘다만족", "하나만만족"]
전략3_청산규칙1_유형 = "빠른선<느린선" # @param ["가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략3_청산규칙1_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략3_청산규칙1_기간1 = 50 # @param {type:"integer"}
전략3_청산규칙1_기간2 = 200 # @param {type:"integer"}
전략3_청산규칙1_기간3 = 0 # @param {type:"integer"}
전략3_청산규칙2_유형 = "사용안함" # @param ["사용안함", "가격>SMA", "가격<SMA", "이평선정배열", "이평선역배열", "빠른선>느린선", "빠른선<느린선", "항상참"]
전략3_청산규칙2_기준 = "매매대상" # @param ["매매대상", "기준시계열", "나스닥100", "S&P500", "합성장기시계열"]
전략3_청산규칙2_기간1 = 0 # @param {type:"integer"}
전략3_청산규칙2_기간2 = 0 # @param {type:"integer"}
전략3_청산규칙2_기간3 = 0 # @param {type:"integer"}
전략3_1차익절_사용 = False # @param {type:"boolean"}
전략3_1차익절_배수 = 1.15 # @param {type:"number"}
전략3_1차익절_매도비율 = 0.5 # @param {type:"number"}
전략3_1차익절_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략3_2차익절_사용 = False # @param {type:"boolean"}
전략3_2차익절_배수 = 1.68 # @param {type:"number"}
전략3_2차익절_매도비율 = 0.35 # @param {type:"number"}
전략3_2차익절_자금이동 = "SPY" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략3_3차익절_사용 = False # @param {type:"boolean"}
전략3_3차익절_배수 = 2.0 # @param {type:"number"}
전략3_3차익절_매도비율 = 1.0 # @param {type:"number"}
전략3_3차익절_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]
전략3_트레일링사용 = False # @param {type:"boolean"}
전략3_트레일링_첫익절후활성 = True # @param {type:"boolean"}
전략3_트레일링_하락폭 = 0.15 # @param {type:"number"}
전략3_트레일링_자금이동 = "현금" # @param ["현금", "SGOV", "SPY", "QQQ", "기준시계열"]


In [ ]:
import json

기준시계열_매핑 = {
    '나스닥100': 'ndx',
    'S&P500': 'spx',
    '합성장기시계열': 'composite_splice',
}

규칙기준_매핑 = {
    '매매대상': 'traded',
    '기준시계열': 'base',
    '나스닥100': 'ndx',
    'S&P500': 'spx',
    '합성장기시계열': 'composite_splice',
}

규칙유형_매핑 = {
    '사용안함': 'none',
    '항상참': 'always_true',
    '가격>SMA': 'price_above_sma',
    '가격<SMA': 'price_below_sma',
    '빠른선>느린선': 'fast_above_slow',
    '빠른선<느린선': 'fast_below_slow',
    '이평선정배열': 'sma_chain_above',
    '이평선역배열': 'sma_chain_below',
}

결합_매핑 = {
    '규칙1만': 'single',
    '둘다만족': 'all_of',
    '하나만만족': 'any_of',
}

자금이동_매핑 = {
    '현금': 'cash',
    'SGOV': 'sgov',
    'SPY': 'spy',
    'QQQ': 'qqq',
    '기준시계열': 'base',
}

def 값(이름):
    return globals()[이름]

def 규칙_만들기(규칙유형, 규칙기준, 기간1, 기간2, 기간3):
    내부유형 = 규칙유형_매핑[규칙유형]
    if 내부유형 == 'none':
        return None
    if 내부유형 == 'always_true':
        return {'type': 'always_true'}

    내부기준 = 규칙기준_매핑[규칙기준]
    if 내부유형 in {'price_above_sma', 'price_below_sma'}:
        return {
            'type': 내부유형,
            'source': 내부기준,
            'window': int(기간1),
        }
    if 내부유형 in {'fast_above_slow', 'fast_below_slow'}:
        return {
            'type': 내부유형,
            'source': 내부기준,
            'fast_window': int(기간1),
            'slow_window': int(기간2),
        }
    if 내부유형 in {'sma_chain_above', 'sma_chain_below'}:
        윈도우목록 = [int(v) for v in (기간1, 기간2, 기간3) if int(v) > 0]
        if len(윈도우목록) < 2:
            raise ValueError(f'{규칙유형}은 기간을 최소 2개 넣어야 합니다.')
        return {
            'type': 내부유형,
            'source': 내부기준,
            'windows': 윈도우목록,
        }
    raise ValueError(f'지원하지 않는 규칙 유형입니다: {규칙유형}')

def 규칙묶기(결합, 규칙1, 규칙2):
    규칙들 = [규칙 for 규칙 in (규칙1, 규칙2) if 규칙 is not None]
    if not 규칙들:
        return {'type': 'always_true'}
    내부결합 = 결합_매핑[결합]
    if 내부결합 == 'single' or len(규칙들) == 1:
        return 규칙들[0]
    return {
        'type': 내부결합,
        'rules': 규칙들,
    }

def 익절리스트_만들기(번호):
    익절목록 = []
    for 단계 in (1, 2, 3):
        if 값(f'전략{번호}_{단계}차익절_사용'):
            익절목록.append({
                'trigger_gain_multiple': float(값(f'전략{번호}_{단계}차익절_배수')),
                'sell_fraction': float(값(f'전략{번호}_{단계}차익절_매도비율')),
                'destination': 자금이동_매핑[값(f'전략{번호}_{단계}차익절_자금이동')],
            })
    익절목록.sort(key=lambda row: row['trigger_gain_multiple'])
    return 익절목록

def 전략_만들기(번호):
    if not 값(f'전략{번호}_사용'):
        return None

    진입규칙1 = 규칙_만들기(
        값(f'전략{번호}_진입규칙1_유형'),
        값(f'전략{번호}_진입규칙1_기준'),
        값(f'전략{번호}_진입규칙1_기간1'),
        값(f'전략{번호}_진입규칙1_기간2'),
        값(f'전략{번호}_진입규칙1_기간3'),
    )
    진입규칙2 = 규칙_만들기(
        값(f'전략{번호}_진입규칙2_유형'),
        값(f'전략{번호}_진입규칙2_기준'),
        값(f'전략{번호}_진입규칙2_기간1'),
        값(f'전략{번호}_진입규칙2_기간2'),
        값(f'전략{번호}_진입규칙2_기간3'),
    )
    청산규칙1 = 규칙_만들기(
        값(f'전략{번호}_청산규칙1_유형'),
        값(f'전략{번호}_청산규칙1_기준'),
        값(f'전략{번호}_청산규칙1_기간1'),
        값(f'전략{번호}_청산규칙1_기간2'),
        값(f'전략{번호}_청산규칙1_기간3'),
    )
    청산규칙2 = 규칙_만들기(
        값(f'전략{번호}_청산규칙2_유형'),
        값(f'전략{번호}_청산규칙2_기준'),
        값(f'전략{번호}_청산규칙2_기간1'),
        값(f'전략{번호}_청산규칙2_기간2'),
        값(f'전략{번호}_청산규칙2_기간3'),
    )

    return {
        'name': 값(f'전략{번호}_이름'),
        'base_series': 기준시계열_매핑[값(f'전략{번호}_기준시계열')],
        'leverage': float(값(f'전략{번호}_레버리지')),
        'include_financing_cost': bool(값(f'전략{번호}_비용반영')),
        'exit_destination': 자금이동_매핑[값(f'전략{번호}_전량청산후자금')],
        'entry': 규칙묶기(값(f'전략{번호}_진입결합'), 진입규칙1, 진입규칙2),
        'exit': 규칙묶기(값(f'전략{번호}_청산결합'), 청산규칙1, 청산규칙2),
        'take_profit_ladder': 익절리스트_만들기(번호),
        'trailing_stop': {
            'enabled': bool(값(f'전략{번호}_트레일링사용')),
            'activate_after_first_take_profit': bool(값(f'전략{번호}_트레일링_첫익절후활성')),
            'drawdown_from_peak': float(값(f'전략{번호}_트레일링_하락폭')),
            'destination': 자금이동_매핑[값(f'전략{번호}_트레일링_자금이동')],
        },
    }

전략목록 = []
for 번호 in (1, 2, 3):
    전략 = 전략_만들기(번호)
    if 전략 is not None:
        전략목록.append(전략)

CONFIG = {
    'start_date': 시작일,
    'end_date': 종료일,
    'commission_rate': float(편도수수료),
    'tax_rate': float(세율),
    'expense_ratio': float(연운용보수),
    'borrow_spread': float(차입스프레드),
    'strategies': 전략목록,
}

print('현재 전략 목록:')
for 전략 in CONFIG['strategies']:
    print('-', 전략['name'])

print('\n생성된 config 미리보기:')
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


In [ ]:
import json
import subprocess
from pathlib import Path

config_path = Path('/content/workbench_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')

subprocess.run(
    ['python', 'run_workbench.py', '--config', str(config_path), '--output-name', 결과폴더이름],
    cwd=repo_dir,
    check=True,
)

print(f'실행 완료: {결과폴더이름}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

output_dir = Path('/root/strategy_workbench_outputs') / 결과폴더이름
metrics = pd.read_csv(output_dir / 'metrics.csv')
display(metrics)

curves = pd.read_csv(output_dir / 'equity_curves.csv')
pivot = curves.pivot(index='date', columns='strategy', values='equity')
pivot.plot(figsize=(12, 6), title='전략 비교 곡선')
plt.ylabel('누적 배수')
plt.show()


In [ ]:
!cd /root && zip -r /content/strategy_workbench_outputs.zip strategy_workbench_outputs

from google.colab import files
files.download('/content/strategy_workbench_outputs.zip')
